## 1. Hugging Face Login

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer
Reuse the byte-level BPE implementation. The first four special tokens stay in their original order; `<mask>` is added as a fifth special token for corruption. The tokenizer is trained only on the training-document split.


In [ ]:
from collections import defaultdict, Counter
import heapq
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>', '<mask>']

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        self.symbol_to_bytes = {}

    def word_to_symbols(self, word):
        byte_seq = word.encode('utf-8')
        return [str(b) for b in byte_seq] + ['</w>']

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print('Building word frequencies...')
        for text in corpus:
            if not isinstance(text, str):
                continue
            word_counter.update(text.split())
        print(f'Unique whitespace-separated words: {len(word_counter):,}')
        word_freqs = {}
        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = word_freqs.get(symbols, 0) + freq
        return word_freqs

    @staticmethod
    def get_pairs(symbols):
        return {(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)}

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[symbols[i], symbols[i + 1]] += 1
        return counts

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and (symbols[i + 1] == pair[1]):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        return tuple(new_symbols)

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = self.specials + self.byte_vocab + ['</w>']
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes['</w>'] = b''
        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        print('Building initial pair statistics...')
        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)
        print(f'Initial unique pairs: {len(pair_counts):,}')
        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap, (-count, pair))
        target_merges = max(0, self.vocab_size - len(vocab))
        print(f'Target merges: {target_merges:,}')
        while len(vocab) < self.vocab_size and heap:
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair, 0)
                if current_count == heap_count and current_count > 0:
                    break
            else:
                break
            max_freq = pair_counts.get(pair, 0)
            if max_freq <= 1:
                break
            merged_symbol = f'<bpe_{len(self.merges)}>'
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes
            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1, merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)
            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                if pair not in self.get_pairs(old_symbols):
                    continue
                old_counts = self.pair_counts_in_word(old_symbols)
                new_symbols = self.merge_symbols(old_symbols, pair, merged_symbol)
                if new_symbols == old_symbols:
                    continue
                new_counts = self.pair_counts_in_word(new_symbols)
                words[word_id] = new_symbols
                all_pairs = set(old_counts) | set(new_counts)
                for p in all_pairs:
                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (new_n - old_n) * freq
                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)
            pair_to_words[pair].clear()
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap, (-count, p))
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {symbol: i for i, symbol in enumerate(vocab)}
        self.itos = {i: symbol for symbol, i in self.stoi.items()}
        print('\nTokenizer training complete.')
        print('Final vocabulary size:', self.vocab_size)
        print('Number of merges:', len(self.merges))

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float('inf')
            best_merged_symbol = None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol
            if best_pair is None:
                break
            symbols = list(self.merge_symbols(tuple(symbols), best_pair, best_merged_symbol))
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.stoi['<s>'])
        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(self.stoi.get(symbol, self.stoi['<unk>']))
        if add_eos:
            ids.append(self.stoi['</s>'])
        return ids

    def decode(self, ids):
        words = []
        current_bytes = bytearray()
        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b'')
            current_bytes.extend(byte_value)
            if ends_word:
                try:
                    word = bytes(current_bytes).decode('utf-8')
                    words.append(word)
                except UnicodeDecodeError:
                    words.append('<unk>')
                current_bytes = bytearray()
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode('utf-8'))
            except UnicodeDecodeError:
                words.append('<unk>')
        return ' '.join(words)

    def _symbol_ends_word(self, symbol):
        if symbol == '</w>':
            return True
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(pair[1])
        return False

    def get_vocab_size(self):
        return len(self.vocab)


## 9. Define The Model
Each block has gated multi-head attention and a GEGLU feed-forward layer, with residual connections. The same weights process both objectives. `causal=False` gives bidirectional encoder attention. `causal=True` gives left-to-right attention. KV caching is allowed only in causal evaluation, because adding a token in bidirectional mode would change earlier representations.


In [ ]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class HybridConfig:
    vocab_size: int
    pad_id: int
    mask_id: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    d_ff: int = 1664
    max_seq_len: int = 512
    dropout: float = 0.1


class GatedAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, key_valid, causal=True, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        past_len = 0 if past is None else past[0].size(2)
        if past is not None:
            k = torch.cat((past[0], k), dim=2)
            v = torch.cat((past[1], v), dim=2)

        allowed = key_valid[:, None, None, :]
        if causal:
            query_positions = torch.arange(past_len, past_len + t, device=x.device)
            key_positions = torch.arange(k.size(2), device=x.device)
            allowed = allowed & (key_positions[None, :] <= query_positions[:, None])[None, None]

        # A fully padded query row gets a harmless fallback key to avoid NaNs.
        first_key = torch.arange(k.size(2), device=x.device).eq(0)
        allowed = allowed | (~allowed.any(dim=-1, keepdim=True) & first_key)
        # Explicit mask also handles offset queries when a KV cache is present.
        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=allowed,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=False,
        )
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        # Paper Section 2.2: GEGLU gate and non-affine output LayerNorm.
        context = context * F.gelu(self.gate(x))
        output = self.out(self.output_norm(context))
        return output, (k, v) if use_cache else None


In [ ]:
class GatedFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_ff, elementwise_affine=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        hidden = self.value(x) * F.gelu(self.gate(x))
        return self.out(self.output_norm(hidden))


class HybridBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.attn = GatedAttention(cfg)
        self.ffn = GatedFeedForward(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, valid, causal, past=None, use_cache=False):
        attention, kv = self.attn(self.norm1(x), valid, causal, past, use_cache)
        x = x + self.dropout(attention)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, kv


class GPTBertLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.embedding_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._initialize)
        self.lm_head.weight = self.tok_emb.weight
        # Smaller residual projections keep a deep stack stable at initialization.
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.ffn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        with torch.no_grad():
            self.tok_emb.weight[cfg.pad_id].zero_()

    def _initialize(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, causal=True, cache=None, use_cache=False, last_only=False):
        if (cache is not None or use_cache) and (not causal or self.training):
            raise ValueError('KV caching is only used in causal evaluation mode.')
        offset = 0 if cache is None else cache['length']
        if offset + ids.size(1) > self.cfg.max_seq_len:
            raise ValueError('Context is full: rebuild from the most recent tokens.')
        positions = torch.arange(offset, offset + ids.size(1), device=ids.device)
        x = self.dropout(self.embedding_norm(self.tok_emb(ids) + self.pos_emb(positions)))
        valid = ids.ne(self.cfg.pad_id)
        if cache is not None:
            valid = torch.cat((cache['valid'], valid), dim=1)
        updated = []
        for i, block in enumerate(self.blocks):
            past = None if cache is None else cache['kv'][i]
            x, kv = block(x, valid, causal, past, use_cache)
            if use_cache:
                updated.append(kv)
        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))
        if use_cache:
            return logits, dict(kv=updated, valid=valid, length=offset + ids.size(1))
        return logits


# **Phase 2**

In [9]:
HYBRID_DIR = "/kaggle/input/datasets/bokkatrishalreddy/saved-files-hybrid"
METHOD = "lora"  # "lora" or "lorma" (LoRMA+)
RANK = 8
ALPHA = 16
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42
RUN_NAME = "qkv_ffn_run1"
MAX_LENGTH = generation_model.cfg.max_seq_len

PHASE2_OUTPUT_DIR = f"/kaggle/working/phase2_hybrid_{METHOD}_{RUN_NAME}"
if os.path.exists(PHASE2_OUTPUT_DIR):
    raise FileExistsError("Choose a new RUN_NAME to preserve the existing run.")
os.makedirs(PHASE2_OUTPUT_DIR)

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Method:", METHOD)
print("Output:", PHASE2_OUTPUT_DIR)

Device: cuda
Method: lora
Output: /kaggle/working/phase2_hybrid_lora_qkv_ffn_run1


In [10]:
import json
from huggingface_hub import hf_hub_download

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

files = [
    "Train/fold1-text.json",
    "Train/fold1-gold.json",
    "Train/fold2-text.json",
    "Train/fold2-gold.json",
    "Train/labels.txt",
    "Test/fold3-text.json",
]

paths = {}

for filename in files:
    paths[filename] = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=f"Phase_2_LNER/{filename}",
        repo_type="dataset",
        token=HF_TOKEN,
    )

import random

fold1_texts = load_json(paths["Train/fold1-text.json"])
fold1_gold = load_json(paths["Train/fold1-gold.json"])

fold2_texts = load_json(paths["Train/fold2-text.json"])
fold2_gold = load_json(paths["Train/fold2-gold.json"])

# Prevent duplicate IDs from silently overwriting documents.
if set(fold1_texts) & set(fold2_texts):
    raise ValueError("The two folds contain duplicate document IDs.")

all_texts = {**fold1_texts, **fold2_texts}
all_gold = {**fold1_gold, **fold2_gold}

if set(all_texts) != set(all_gold):
    raise ValueError("Document IDs and annotation IDs do not match.")

# Fixed seed keeps the split identical across experiments.
doc_ids = sorted(all_texts)
random.Random(42).shuffle(doc_ids)

split_index = int(0.9 * len(doc_ids))

train_ids = doc_ids[:split_index]
val_ids = doc_ids[split_index:]

train_texts = {doc_id: all_texts[doc_id] for doc_id in train_ids}
train_gold = {doc_id: all_gold[doc_id] for doc_id in train_ids}

val_texts = {doc_id: all_texts[doc_id] for doc_id in val_ids}
val_gold = {doc_id: all_gold[doc_id] for doc_id in val_ids}

print("Training documents:", len(train_texts))
print("Validation documents:", len(val_texts))

test_texts = load_json(paths["Test/fold3-text.json"])

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Training documents: 63
Validation documents: 7


In [11]:
tokenizer = generation_tokenizer

In [12]:
with open(paths["Train/labels.txt"], encoding="utf-8") as f:
    entity_types = [line.strip() for line in f if line.strip()]

tags = ["O"]

for entity in entity_types:
    tags.append(f"B-{entity}")
    tags.append(f"I-{entity}")

label2id = {tag: i for i, tag in enumerate(tags)}
id2label = {i: tag for tag, i in label2id.items()}

print("Number of labels:", len(tags))  # 25
print(label2id)

Number of labels: 25
{'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4, 'B-A.COUNSEL': 5, 'I-A.COUNSEL': 6, 'B-R.COUNSEL': 7, 'I-R.COUNSEL': 8, 'B-JUDGE': 9, 'I-JUDGE': 10, 'B-WIT': 11, 'I-WIT': 12, 'B-AUTH': 13, 'I-AUTH': 14, 'B-COURT': 15, 'I-COURT': 16, 'B-STAT': 17, 'I-STAT': 18, 'B-PREC': 19, 'I-PREC': 20, 'B-DATE': 21, 'I-DATE': 22, 'B-CASENO': 23, 'I-CASENO': 24}


In [ ]:
import re
import string


def encode_with_offsets(text, tokenizer):
    input_ids = []
    offsets = []

    punctuation = set(string.punctuation.encode("ascii"))

    # Process both whitespace and non-whitespace segments.
    for match in re.finditer(r"\s+|\S+", text):
        segment = match.group()
        segment_start = match.start()

        # Preserve whitespace and its exact character positions.
        if segment.isspace():
            for char_index, char in enumerate(segment):
                start = segment_start + char_index

                for byte in char.encode("utf-8"):
                    input_ids.append(tokenizer.stoi[str(byte)])
                    offsets.append((start, start + 1))

            continue

        # Map each UTF-8 byte to its original character position.
        byte_to_char = []

        for char_index, char in enumerate(segment):
            byte_to_char.extend(
                [segment_start + char_index]
                * len(char.encode("utf-8"))
            )

        byte_position = 0

        for symbol in tokenizer.apply_bpe(segment):
            token_id = tokenizer.stoi[symbol]
            token_bytes = tokenizer.symbol_to_bytes[symbol]

            split_punctuation = (
                len(token_bytes) > 1
                and any(byte in punctuation for byte in token_bytes)
            )

            split_boundary = token_id in split_token_ids

            if token_bytes and (
                split_punctuation or split_boundary
            ):
                pieces = [str(byte) for byte in token_bytes]

                if tokenizer._symbol_ends_word(symbol):
                    pieces.append("</w>")
            else:
                pieces = [symbol]

            for piece in pieces:
                byte_length = len(
                    tokenizer.symbol_to_bytes[piece]
                )

                input_ids.append(tokenizer.stoi[piece])

                if byte_length == 0:
                    offsets.append(None)
                else:
                    start = byte_to_char[byte_position]
                    end = (
                        byte_to_char[
                            byte_position + byte_length - 1
                        ] + 1
                    )

                    offsets.append((start, end))

                byte_position += byte_length

        if byte_position != len(byte_to_char):
            raise ValueError(
                f"Incomplete byte coverage for {segment!r}"
            )

    return input_ids, offsets

In [14]:
def make_ner_labels(offsets, annotations, label2id):
    labels = [
        -100 if offset is None else label2id["O"]
        for offset in offsets
    ]

    boundary_cases = []

    for start, end, entity_type, *_ in sorted(annotations):
        if start < 0 or end <= start:
            raise ValueError("Invalid gold span.")
        indices = [
            i
            for i, offset in enumerate(offsets)
            if offset is not None
            and offset[0] < end
            and offset[1] > start
        ]

        if not indices:
            raise ValueError(
                f"No tokens overlap {(start, end, entity_type)}"
            )

        if any(labels[i] != label2id["O"] for i in indices):
            raise ValueError(
                f"Two entities share tokens near "
                f"{(start, end, entity_type)}"
            )

        first, last = indices[0], indices[-1]
        token_span = (offsets[first][0], offsets[last][1])

        if token_span != (start, end):
            boundary_cases.append({
                "gold_span": (start, end),
                "token_span": token_span,
                "label": entity_type,
            })

        labels[first] = label2id[f"B-{entity_type}"]

        for i in indices[1:]:
            labels[i] = label2id[f"I-{entity_type}"]

    return labels, boundary_cases

In [15]:
# # Learn splitting rules from TRAINING annotations only.
# split_token_ids = set()

# for doc_id, record in encoded_train.items():
#     boundaries = {
#         boundary
#         for annotation in train_gold[doc_id]
#         for boundary in annotation[:2]
#     }

#     for token_id, offset in zip(
#         record["input_ids"],
#         record["offsets"],
#     ):
#         if offset is None:
#             continue

#         start, end = offset

#         if any(start < boundary < end for boundary in boundaries):
#             split_token_ids.add(token_id)

# print("Token IDs requiring finer boundaries:", len(split_token_ids))

# for token_id in sorted(split_token_ids):
#     symbol = tokenizer.itos[token_id]
#     print(token_id, repr(tokenizer.symbol_to_bytes[symbol]))

split_token_ids = {
    559,885,1049,1230,1245,1710,3352, 9881,
}

In [16]:
def encode_documents(texts, annotations):
    encoded = {}

    for doc_id, text in texts.items():
        input_ids, offsets = encode_with_offsets(text, tokenizer)

        labels, boundary_cases = make_ner_labels(
            offsets,
            annotations[doc_id],
            label2id,
        )

        encoded[doc_id] = {
            "input_ids": input_ids,
            "labels": labels,
            "offsets": offsets,
            "boundary_cases": boundary_cases,
        }

    return encoded


encoded_train = encode_documents(train_texts, train_gold)
encoded_val = encode_documents(val_texts, val_gold)

In [17]:
import pandas as pd


def analyze_gold_token_alignment(texts, gold, encoded_documents):
    records = []

    for doc_id, text in texts.items():
        offsets = encoded_documents[doc_id]["offsets"]

        for entity_idx, annotation in enumerate(gold[doc_id]):
            gold_start, gold_end, label, *_ = annotation

            overlapping = [
                (i, offset[0], offset[1])
                for i, offset in enumerate(offsets)
                if offset is not None
                and offset[0] < gold_end
                and offset[1] > gold_start
            ]

            row = {
                "doc_id": doc_id,
                "entity_idx": entity_idx,
                "label": label,
                "gold_text": text[gold_start:gold_end],
                "gold_start": gold_start,
                "gold_end": gold_end,
                "first_token_idx": None,
                "last_token_idx": None,
                "token_start": None,
                "token_end": None,
                "start_diff": None,
                "end_diff": None,
                "prefix_extra": "",
                "suffix_extra": "",
                "token_text": "",
                "alignment_type": "NO_TOKEN",
            }

            if overlapping:
                first_idx = overlapping[0][0]
                last_idx = overlapping[-1][0]

                token_start = min(item[1] for item in overlapping)
                token_end = max(item[2] for item in overlapping)

                if token_start == gold_start and token_end == gold_end:
                    alignment_type = "EXACT"

                elif token_start == gold_start and token_end > gold_end:
                    alignment_type = "SUFFIX_ONLY"

                elif token_start < gold_start and token_end == gold_end:
                    alignment_type = "PREFIX_ONLY"

                elif token_start < gold_start and token_end > gold_end:
                    alignment_type = "PREFIX_AND_SUFFIX"

                elif token_start > gold_start and token_end < gold_end:
                    alignment_type = "INSIDE_GOLD"

                elif token_start > gold_start:
                    alignment_type = "START_MISMATCH"

                elif token_end < gold_end:
                    alignment_type = "END_MISMATCH"

                else:
                    alignment_type = "OTHER"

                row.update({
                    "first_token_idx": first_idx,
                    "last_token_idx": last_idx,
                    "token_start": token_start,
                    "token_end": token_end,
                    "start_diff": token_start - gold_start,
                    "end_diff": token_end - gold_end,
                    "prefix_extra": (
                        text[token_start:gold_start]
                        if token_start < gold_start else ""
                    ),
                    "suffix_extra": (
                        text[gold_end:token_end]
                        if token_end > gold_end else ""
                    ),
                    "token_text": text[token_start:token_end],
                    "alignment_type": alignment_type,
                })

            records.append(row)

    if not records:
        raise ValueError("No gold entities found.")

    all_df = pd.DataFrame(records)

    alignment_counts = (
        all_df["alignment_type"].value_counts().to_dict()
    )

    total = len(all_df)
    exact = alignment_counts.get("EXACT", 0)

    summary = {
        "total_gold_entities": total,
        "exact": exact,
        "mismatched": total - exact,
        "exact_percentage": 100 * exact / total,
        "mismatch_percentage": 100 * (total - exact) / total,
        "alignment_counts": alignment_counts,
    }

    entity_stats = (
        all_df.groupby("label")
        .agg(
            total=("alignment_type", "size"),
            exact=("alignment_type", lambda x: (x == "EXACT").sum()),
            suffix_only=(
                "alignment_type",
                lambda x: (x == "SUFFIX_ONLY").sum(),
            ),
            prefix_only=(
                "alignment_type",
                lambda x: (x == "PREFIX_ONLY").sum(),
            ),
            prefix_suffix=(
                "alignment_type",
                lambda x: (x == "PREFIX_AND_SUFFIX").sum(),
            ),
            mismatched=(
                "alignment_type",
                lambda x: (x != "EXACT").sum(),
            ),
        )
        .reset_index()
    )

    entity_stats["exact_pct"] = (
        100 * entity_stats["exact"] / entity_stats["total"]
    )

    entity_stats["mismatch_pct"] = (
        100 * entity_stats["mismatched"] / entity_stats["total"]
    )

    entity_stats = entity_stats.sort_values(
        "mismatch_pct", ascending=False
    )

    mismatches_df = all_df[
        all_df["alignment_type"] != "EXACT"
    ].copy()

    return summary, entity_stats, mismatches_df, all_df

In [18]:
summary, entity_stats, mismatches_df, all_df = (
    analyze_gold_token_alignment(
        texts=train_texts,
        gold=train_gold,
        encoded_documents=encoded_train,
    )
)

print(summary)
display(entity_stats)

display(
    mismatches_df[
        [
            "doc_id",
            "label",
            "gold_text",
            "token_text",
            "prefix_extra",
            "suffix_extra",
            "alignment_type",
        ]
    ].head(30)
)

{'total_gold_entities': 7714, 'exact': 7714, 'mismatched': 0, 'exact_percentage': 100.0, 'mismatch_percentage': 0.0, 'alignment_counts': {'EXACT': 7714}}


,label,total,exact,suffix_only,prefix_only,prefix_suffix,mismatched,exact_pct,mismatch_pct
0,A.COUNSEL,168,168,0,0,0,0,100.0,0.0
1,APP,390,390,0,0,0,0,100.0,0.0
2,AUTH,912,912,0,0,0,0,100.0,0.0
3,CASENO,695,695,0,0,0,0,100.0,0.0
4,COURT,939,939,0,0,0,0,100.0,0.0
5,DATE,1446,1446,0,0,0,0,100.0,0.0
6,JUDGE,226,226,0,0,0,0,100.0,0.0
7,PREC,1064,1064,0,0,0,0,100.0,0.0
8,R.COUNSEL,162,162,0,0,0,0,100.0,0.0
9,RESP,362,362,0,0,0,0,100.0,0.0


,doc_id,label,gold_text,token_text,prefix_extra,suffix_extra,alignment_type


In [19]:
val_summary, val_entity_stats, val_mismatches_df, val_all_df = (
    analyze_gold_token_alignment(
        texts=val_texts,
        gold=val_gold,
        encoded_documents=encoded_val,
    )
)

print(val_summary)
display(val_entity_stats)
display(val_mismatches_df.head(30))

{'total_gold_entities': 679, 'exact': 679, 'mismatched': 0, 'exact_percentage': 100.0, 'mismatch_percentage': 0.0, 'alignment_counts': {'EXACT': 679}}


,label,total,exact,suffix_only,prefix_only,prefix_suffix,mismatched,exact_pct,mismatch_pct
0,A.COUNSEL,27,27,0,0,0,0,100.0,0.0
1,APP,17,17,0,0,0,0,100.0,0.0
2,AUTH,91,91,0,0,0,0,100.0,0.0
3,CASENO,65,65,0,0,0,0,100.0,0.0
4,COURT,74,74,0,0,0,0,100.0,0.0
5,DATE,129,129,0,0,0,0,100.0,0.0
6,JUDGE,19,19,0,0,0,0,100.0,0.0
7,PREC,119,119,0,0,0,0,100.0,0.0
8,R.COUNSEL,22,22,0,0,0,0,100.0,0.0
9,RESP,19,19,0,0,0,0,100.0,0.0


,doc_id,entity_idx,label,gold_text,gold_start,gold_end,first_token_idx,last_token_idx,token_start,token_end,start_diff,end_diff,prefix_extra,suffix_extra,token_text,alignment_type


In [20]:
MAX_LENGTH = 512
LEFT_CONTEXT = 128
RIGHT_CONTEXT = 128


def create_windows(
    documents,
    max_length=MAX_LENGTH,
    left_context=LEFT_CONTEXT,
    right_context=RIGHT_CONTEXT,
):
    middle_length = max_length - left_context - right_context

    if min(left_context, right_context) < 0 or middle_length <= 0:
        raise ValueError("Invalid context lengths.")

    windows = []

    for doc_id, record in documents.items():
        total_tokens = len(record["input_ids"])

        for prediction_start in range(0, total_tokens, middle_length):
            prediction_end = min(
                prediction_start + middle_length,
                total_tokens,
            )

            start = max(0, prediction_start - left_context)
            end = min(total_tokens, prediction_end + right_context)

            # Positions relative to this window.
            target_start = prediction_start - start
            target_end = prediction_end - start

            labels = list(record["labels"][start:end])

            labels[:target_start] = [-100] * target_start
            labels[target_end:] = [-100] * (len(labels) - target_end)

            windows.append({
                "doc_id": doc_id,
                "token_start": start,
                "target_start": target_start,
                "target_end": target_end,
                "input_ids": record["input_ids"][start:end],
                "labels": labels,
                "offsets": record["offsets"][start:end],
            })

    return windows


train_windows = create_windows(encoded_train)
val_windows = create_windows(encoded_val)

In [21]:
NUM_LABELS = len(label2id)
counts = torch.zeros(NUM_LABELS)

for example in train_windows:
    for label_id in example["labels"]:
        if label_id != -100:
            counts[label_id] += 1

total = counts.sum()

weights = torch.ones(NUM_LABELS)
entity_ids = []

for entity_type in entity_types:
    b_id = label2id[f"B-{entity_type}"]
    i_id = label2id[f"I-{entity_type}"]

    count = counts[b_id] + counts[i_id]
    class_weight = (total / count.clamp(min=1)) ** 0.25

    weights[b_id] = class_weight *1.5
    weights[i_id] = class_weight

    entity_ids.extend([b_id, i_id])

weights[entity_ids] /= weights[entity_ids].mean()
weights[label2id["O"]] = 0.3

for i in range(NUM_LABELS):
    print(f"{id2label[i]:15s} {weights[i].item():.4f}")

O               0.3000
B-APP           1.2651
I-APP           0.8434
B-RESP          1.2914
I-RESP          0.8609
B-A.COUNSEL     1.5651
I-A.COUNSEL     1.0434
B-R.COUNSEL     1.5738
I-R.COUNSEL     1.0492
B-JUDGE         1.3619
I-JUDGE         0.9079
B-WIT           1.6400
I-WIT           1.0933
B-AUTH          1.1230
I-AUTH          0.7487
B-COURT         1.1148
I-COURT         0.7432
B-STAT          0.9218
I-STAT          0.6146
B-PREC          0.7445
I-PREC          0.4963
B-DATE          0.8803
I-DATE          0.5869
B-CASENO        0.9183
I-CASENO        0.6122


In [22]:
for name, documents, windows in [
    ("Train", encoded_train, train_windows),
    ("Validation", encoded_val, val_windows),
]:
    boundary_count = sum(
        len(record["boundary_cases"])
        for record in documents.values()
    )

    print(
        f"{name}: {len(documents)} documents, "
        f"{len(windows)} windows, "
        f"{boundary_count} approximate entity boundaries"
    )

Train: 63 documents, 3713 windows, 0 approximate entity boundaries
Validation: 7 documents, 335 windows, 0 approximate entity boundaries


In [23]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

pad_id = tokenizer.stoi["<pad>"]

def collate_batch(examples):
    input_ids = [
        torch.tensor(example["input_ids"], dtype=torch.long)
        for example in examples
    ]

    labels = [
        torch.tensor(example["labels"], dtype=torch.long)
        for example in examples
    ]

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=pad_id,
        ),
        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        ),
        "doc_ids": [example["doc_id"] for example in examples],
        "token_starts": [
            example["token_start"] for example in examples
        ],
        "target_starts": [
            example["target_start"] for example in examples
        ],
        "target_ends": [
            example["target_end"] for example in examples
        ],
        "offsets": [example["offsets"] for example in examples],
    }

In [24]:
train_loader = DataLoader(
    train_windows,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_windows,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

In [25]:
batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Label shape:", batch["labels"].shape)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Input shape: torch.Size([16, 512])
Label shape: torch.Size([16, 512])
Training batches: 233
Validation batches: 21


# **Model Creation**

In [26]:
class HybridNER(nn.Module):
    def __init__(self, pretrained_model, num_labels):
        super().__init__()
        self.cfg = pretrained_model.cfg
        self.tok_emb = pretrained_model.tok_emb
        self.pos_emb = pretrained_model.pos_emb
        self.embedding_norm = pretrained_model.embedding_norm
        self.dropout = pretrained_model.dropout
        self.blocks = pretrained_model.blocks
        self.final_norm = pretrained_model.final_norm
        self.classifier = nn.Linear(self.cfg.d_model, num_labels)

    def forward(self, input_ids):
        length = input_ids.size(1)
        if length > self.cfg.max_seq_len:
            raise ValueError("Input exceeds the learned position limit.")
        positions = torch.arange(length, device=input_ids.device)
        x = self.tok_emb(input_ids) + self.pos_emb(positions)
        x = self.dropout(self.embedding_norm(x))
        valid = input_ids.ne(self.cfg.pad_id)
        for block in self.blocks:
            x, _ = block(x, valid, causal=False, past=None, use_cache=False)
        return self.classifier(self.final_norm(x))

**LORMA**

In [27]:
class LoRMAPlus(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)

        out_dim = base_layer.out_features
        self.scale = alpha / rank

        self.A = nn.Linear(out_dim, rank, bias=False)
        self.B = nn.Linear(rank, out_dim, bias=False)

        self.A.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )
        self.B.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(original))

        return original + self.scale * correction

**LORA**

In [28]:
class LoRA(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)

        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
        )

        self.B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
        )

        self.A.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )
        self.B.to(
            device=base_layer.weight.device,
            dtype=base_layer.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)
        correction = self.B(self.A(x))

        return original + self.scale * correction

In [29]:
ATTENTION_TARGETS = ["qkv", "out", "gate"]
FFN_TARGETS = ["value", "gate", "out"]

adapter_classes = {
    "lora": LoRA,
    "lorma": LoRMAPlus,
}

if METHOD not in adapter_classes:
    raise ValueError("METHOD must be 'lora' or 'lorma'.")

Adapter = adapter_classes[METHOD]


# Load fresh Phase 1 weights
checkpoint = torch.load(
    os.path.join(HYBRID_DIR, "model.pt"),
    map_location="cpu",
    weights_only=True,
)

cfg = HybridConfig(**checkpoint["model_config"])

if MAX_LENGTH > cfg.max_seq_len:
    raise ValueError(
        "MAX_LENGTH exceeds the checkpoint's position limit."
    )

phase1_model = GPTBertLM(cfg)

phase1_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

del checkpoint


# Initialize a fresh NER classifier
torch.manual_seed(SEED)

ner_model = HybridNER(
    phase1_model,
    num_labels=len(label2id),
)

del phase1_model


# Freeze pretrained parameters
ner_model.requires_grad_(False)


# Add attention and FFN adapters
for block in ner_model.blocks:

    for name in ATTENTION_TARGETS:
        layer = getattr(block.attn, name)

        setattr(
            block.attn,
            name,
            Adapter(
                layer,
                rank=RANK,
                alpha=ALPHA,
            ),
        )

    for name in FFN_TARGETS:
        layer = getattr(block.ffn, name)

        setattr(
            block.ffn,
            name,
            Adapter(
                layer,
                rank=RANK,
                alpha=ALPHA,
            ),
        )


# Train the classifier too
ner_model.classifier.requires_grad_(True)
ner_model = ner_model.to(device)


adaptation_config = {
    "adapter_type": METHOD,
    "rank": RANK,
    "alpha": ALPHA,
    "attention_targets": ATTENTION_TARGETS.copy(),
    "ffn_targets": FFN_TARGETS.copy(),
    "ner_causal": False,
}


total = sum(
    p.numel()
    for p in ner_model.parameters()
)

trainable = sum(
    p.numel()
    for p in ner_model.parameters()
    if p.requires_grad
)

print("Method:", METHOD)
print("Attention targets:", ATTENTION_TARGETS)
print("FFN targets:", FFN_TARGETS)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Method: lora
Attention targets: ['qkv', 'out', 'gate']
FFN targets: ['value', 'gate', 'out']
Total parameters: 52,814,361
Trainable parameters: 1,032,729
Trainable percentage: 1.96%


In [30]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)

ner_model.eval()

with torch.no_grad():
    logits = ner_model(input_ids)

    loss = criterion(
        logits.reshape(-1, len(label2id)),
        labels.reshape(-1),
    )

print("Input shape:", input_ids.shape)
print("Output shape:", logits.shape)
print("Label shape:", labels.shape)
print("Initial loss:", loss.item())

del input_ids, labels, logits, loss

Input shape: torch.Size([16, 512])
Output shape: torch.Size([16, 512, 25])
Label shape: torch.Size([16, 512])
Initial loss: 3.477330446243286


# **Training**

In [31]:
def decode_entities(predicted_ids, offsets):
    entities = []
    current = None

    for label_id, offset in zip(predicted_ids, offsets):
        if offset is None:
            continue

        tag = id2label[label_id]
        start, end = offset

        if tag == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = tag.split("-", 1)

        if (
            prefix == "I"
            and current is not None
            and current[2] == entity_type
        ):
            current[1] = end
        else:
            if current is not None:
                entities.append(current)

            current = [start, end, entity_type]

    if current is not None:
        entities.append(current)

    return entities

In [32]:
import numpy as np


def viterbi_bio_decode(logits, id2label):
    # logits: [number_of_tokens, number_of_labels]
    length, num_labels = logits.shape

    if length == 0:
        return []

    # Rows = previous tag, columns = current tag.
    transitions = np.zeros(
        (num_labels, num_labels),
        dtype=np.float32,
    )

    start_scores = np.zeros(num_labels, dtype=np.float32)

    for current_id in range(num_labels):
        tag = id2label[current_id]

        if tag.startswith("I-"):
            entity_type = tag[2:]

            # An I-tag cannot start a document.
            start_scores[current_id] = -np.inf

            # An I-X tag must follow B-X or I-X.
            for previous_id in range(num_labels):
                previous_tag = id2label[previous_id]

                if previous_tag not in (
                    f"B-{entity_type}",
                    f"I-{entity_type}",
                ):
                    transitions[previous_id, current_id] = -np.inf

    scores = logits[0] + start_scores

    backpointers = np.zeros(
        (length, num_labels),
        dtype=np.int32,
    )

    for position in range(1, length):
        candidates = scores[:, None] + transitions

        best_previous = candidates.argmax(axis=0)
        backpointers[position] = best_previous

        scores = (
            candidates[best_previous, np.arange(num_labels)]
            + logits[position]
        )

    # Recover the best complete sequence.
    best_last = int(scores.argmax())
    predictions = [best_last]

    for position in range(length - 1, 0, -1):
        best_last = int(backpointers[position, best_last])
        predictions.append(best_last)

    predictions.reverse()
    return predictions

In [33]:
@torch.no_grad()
def predict_entities(
    model,
    loader,
    encoded_documents,
    use_viterbi=True,
):
    model.eval()
    device = next(model.parameters()).device

    document_logits = {
        doc_id: np.zeros(
            (len(record["input_ids"]), len(id2label)),
            dtype=np.float32,
        )
        for doc_id, record in encoded_documents.items()
    }

    covered = {
        doc_id: np.zeros(len(record["input_ids"]), dtype=bool)
        for doc_id, record in encoded_documents.items()
    }

    for batch in loader:
        inputs = batch["input_ids"].to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(inputs)

        logits = logits.float().cpu().numpy()

        for i, doc_id in enumerate(batch["doc_ids"]):
            window_start = batch["token_starts"][i]

            # Middle-region positions inside the window.
            local_start = batch["target_starts"][i]
            local_end = batch["target_ends"][i]

            # Corresponding positions inside the document.
            start = window_start + local_start
            end = window_start + local_end

            document_logits[doc_id][start:end] = (
                logits[i, local_start:local_end]
            )

            covered[doc_id][start:end] = True

    results = {}

    for doc_id, record in encoded_documents.items():
        if not covered[doc_id].all():
            raise ValueError(
                f"Missing predictions for document {doc_id}"
            )

        valid_positions = [
            i
            for i, offset in enumerate(record["offsets"])
            if offset is not None
        ]

        valid_logits = document_logits[doc_id][valid_positions]

        valid_offsets = [
            record["offsets"][i]
            for i in valid_positions
        ]

        if use_viterbi:
            predicted_ids = viterbi_bio_decode(
                valid_logits,
                id2label,
            )
        else:
            predicted_ids = valid_logits.argmax(axis=-1).tolist()

        results[doc_id] = decode_entities(
            predicted_ids,
            valid_offsets,
        )

    return results

In [34]:
def strict_ner_scores(predictions, gold):
    if set(predictions) != set(gold):
        raise ValueError("Prediction and gold document IDs do not match.")

    per_label = {}

    for entity_type in entity_types:
        predicted = {
            (doc_id, start, end)
            for doc_id, entities in predictions.items()
            for start, end, label in entities
            if label == entity_type
        }

        actual = {
            (doc_id, annotation[0], annotation[1])
            for doc_id, entities in gold.items()
            for annotation in entities
            if annotation[2] == entity_type
        }

        correct = len(predicted & actual)

        precision = correct / len(predicted) if predicted else 0.0
        recall = correct / len(actual) if actual else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[entity_type] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": len(actual),
        }

    scores = {
        metric: sum(row[metric] for row in per_label.values())
        / len(entity_types)
        for metric in ("precision", "recall", "f1")
    }

    return scores, per_label

In [35]:
import os
import torch
import torch.nn.functional as F
from dataclasses import asdict


@torch.no_grad()
def evaluate_ner_loss(model, loader):
    model.eval()

    device = next(model.parameters()).device
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        inputs = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model(inputs)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum",
            )

        total_loss += loss.item()
        total_tokens += labels.ne(-100).sum().item()

    if total_tokens == 0:
        raise ValueError("No labelled tokens in the validation loader.")

    return total_loss / total_tokens

In [36]:
def train_ner(model, train_loader, val_loader, epochs, lr, output_dir, patience=5):
    os.makedirs(output_dir, exist_ok=True)

    device = next(model.parameters()).device

    trainable_params = [
        p for p in model.parameters() if p.requires_grad
    ]

    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )

    history = []
    best_val_loss = float("inf")

    # device = next(model.parameters()).device
    class_weights = weights.to(device)

    best_macro_f1 = -1.0
    count=0


    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0.0
        total_tokens = 0

        for step, batch in enumerate(train_loader, start=1):
            inputs = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            token_count = labels.ne(-100).sum().item()

            if token_count == 0:
                continue

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=device.type == "cuda",
            ):
                logits = model(inputs)

                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1),
                    weight=class_weights,
                    ignore_index=-100,
                )

            if not torch.isfinite(loss):
                raise ValueError("Non-finite training loss.")

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * token_count
            total_tokens += token_count

            if step % 50 == 0:
                print(
                    f"Epoch {epoch} | Batch {step}/{len(train_loader)} "
                    f"| Loss: {loss.item():.4f}",
                    flush=True,
                )

        if total_tokens == 0:
            raise ValueError("No labelled tokens in the training loader.")

        train_loss = total_loss / total_tokens
        val_loss = evaluate_ner_loss(model, val_loader)

        val_predictions = predict_entities(
            model,
            val_loader,
            encoded_val,
        )
        scores, per_label = strict_ner_scores(
            val_predictions,
            val_gold,
        )
        scheduler.step(scores["f1"])

        # history.append({
        #     "epoch": epoch,
        #     "train_loss": train_loss,
        #     "val_loss": val_loss,
        # })
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "macro_f1": scores["f1"],
            "next_lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"Epoch {epoch} | Train: {train_loss:.4f} "
            f"| Validation: {val_loss:.4f}",
            flush=True,
        )

        # Save every epoch for later comparison using strict Macro-F1.
        checkpoint = {
            "model_state_dict": {
                name: value.detach().cpu()
                for name, value in model.state_dict().items()
            },
            "model_config": asdict(model.cfg),
            "label2id": label2id,
            "adaptation_config": adaptation_config,
            "epoch": epoch,
            "val_loss": val_loss,
            "validation_scores": scores,
        }

        torch.save(
            checkpoint,
            os.path.join(output_dir, f"epoch_{epoch}.pt"),
        )

        # if val_loss < best_val_loss:
        #     best_val_loss = val_loss

        #     torch.save(
        #         checkpoint,
        #         os.path.join(output_dir, "best_loss.pt"),
        #     )
        #     count=0
        # else:
        #     count+=1
        #     if count==patience:
        #         break
        if scores["f1"] > best_macro_f1:
            best_macro_f1 = scores["f1"]
            count = 0
        else:
            count += 1
        
        if count >= patience:
            break

    with open(os.path.join(output_dir, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    return history, epoch

In [37]:
torch.manual_seed(SEED)

history, last_epoch = train_ner(
    ner_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=PHASE2_OUTPUT_DIR,
    patience=10,
)

Epoch 1 | Batch 50/233 | Loss: 0.8337
Epoch 1 | Batch 100/233 | Loss: 0.7235
Epoch 1 | Batch 150/233 | Loss: 0.8230
Epoch 1 | Batch 200/233 | Loss: 0.3857
Epoch 1 | Train: 0.9372 | Validation: 0.3552
Epoch 2 | Batch 50/233 | Loss: 0.5342
Epoch 2 | Batch 100/233 | Loss: 0.4839
Epoch 2 | Batch 150/233 | Loss: 0.8169
Epoch 2 | Batch 200/233 | Loss: 0.5076
Epoch 2 | Train: 0.5898 | Validation: 0.3344
Epoch 3 | Batch 50/233 | Loss: 0.6561
Epoch 3 | Batch 100/233 | Loss: 0.5655
Epoch 3 | Batch 150/233 | Loss: 0.5152
Epoch 3 | Batch 200/233 | Loss: 0.6002
Epoch 3 | Train: 0.5085 | Validation: 0.3016
Epoch 4 | Batch 50/233 | Loss: 0.1922
Epoch 4 | Batch 100/233 | Loss: 0.4756
Epoch 4 | Batch 150/233 | Loss: 0.5683
Epoch 4 | Batch 200/233 | Loss: 0.3813
Epoch 4 | Train: 0.4687 | Validation: 0.2933
Epoch 5 | Batch 50/233 | Loss: 0.3545
Epoch 5 | Batch 100/233 | Loss: 0.4647
Epoch 5 | Batch 150/233 | Loss: 0.4194
Epoch 5 | Batch 200/233 | Loss: 0.4636
Epoch 5 | Train: 0.4434 | Validation: 0.3176


In [38]:
best_f1 = -1.0
best_epoch = None
best_per_label = None
evaluation_history = []

for epoch in range(1, last_epoch+1):
    checkpoint_path = os.path.join(
        PHASE2_OUTPUT_DIR,
        f"epoch_{epoch}.pt",
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )

    ner_model.load_state_dict(checkpoint["model_state_dict"])

    val_predictions = predict_entities(
        ner_model,
        val_loader,
        encoded_val,
    )

    scores, per_label = strict_ner_scores(
        val_predictions,
        val_gold,
    )

    evaluation_history.append({
        "epoch": epoch,
        **scores,
    })

    print(
        f"Epoch {epoch} | "
        f"Precision: {scores['precision']:.4f} | "
        f"Recall: {scores['recall']:.4f} | "
        f"Macro-F1: {scores['f1']:.4f}"
    )

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        best_epoch = epoch
        best_per_label = per_label

        checkpoint["validation_scores"] = scores

        torch.save(
            checkpoint,
            os.path.join(PHASE2_OUTPUT_DIR, "best_f1.pt"),
        )

        with open(
            os.path.join(PHASE2_OUTPUT_DIR, "validation_predictions.json"),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(val_predictions, f, ensure_ascii=False, indent=2)

    del checkpoint

with open(
    os.path.join(PHASE2_OUTPUT_DIR, "evaluation_history.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(evaluation_history, f, indent=2)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation Macro-F1: {best_f1:.4f}")

for label, row in best_per_label.items():
    print(
        f"{label:10s} "
        f"F1: {row['f1']:.4f} | "
        f"Gold entities: {row['support']}"
    )

Epoch 1 | Precision: 0.0269 | Recall: 0.0171 | Macro-F1: 0.0204
Epoch 2 | Precision: 0.0220 | Recall: 0.0530 | Macro-F1: 0.0291
Epoch 3 | Precision: 0.0308 | Recall: 0.0549 | Macro-F1: 0.0380
Epoch 4 | Precision: 0.0343 | Recall: 0.0595 | Macro-F1: 0.0413
Epoch 5 | Precision: 0.0345 | Recall: 0.0795 | Macro-F1: 0.0474
Epoch 6 | Precision: 0.0312 | Recall: 0.0560 | Macro-F1: 0.0399
Epoch 7 | Precision: 0.0354 | Recall: 0.0816 | Macro-F1: 0.0489
Epoch 8 | Precision: 0.0336 | Recall: 0.0757 | Macro-F1: 0.0457
Epoch 9 | Precision: 0.0516 | Recall: 0.1284 | Macro-F1: 0.0708
Epoch 10 | Precision: 0.0531 | Recall: 0.1039 | Macro-F1: 0.0687
Epoch 11 | Precision: 0.0495 | Recall: 0.0910 | Macro-F1: 0.0632
Epoch 12 | Precision: 0.0339 | Recall: 0.0809 | Macro-F1: 0.0472
Epoch 13 | Precision: 0.0488 | Recall: 0.1039 | Macro-F1: 0.0653
Epoch 14 | Precision: 0.0411 | Recall: 0.0979 | Macro-F1: 0.0569
Epoch 15 | Precision: 0.0533 | Recall: 0.1099 | Macro-F1: 0.0701
Epoch 16 | Precision: 0.0503 | Rec

# **Testing**

In [39]:
checkpoint = torch.load(
    os.path.join(PHASE2_OUTPUT_DIR, "best_f1.pt"),
    map_location="cpu",
    weights_only=True,
)

ner_model.load_state_dict(checkpoint["model_state_dict"])

label2id = checkpoint["label2id"]
id2label = {i: label for label, i in label2id.items()}

ner_model.eval()

print("Loaded epoch:", checkpoint["epoch"])

del checkpoint

Loaded epoch: 9


In [40]:
test_texts = load_json(paths["Test/fold3-text.json"])

encoded_test = {}

for doc_id, text in test_texts.items():
    input_ids, offsets = encode_with_offsets(text, tokenizer)

    encoded_test[doc_id] = {
        "input_ids": input_ids,
        "offsets": offsets,
        "labels": [-100] * len(input_ids),
    }

test_windows = create_windows(encoded_test)

test_loader = DataLoader(
    test_windows,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

print("Test documents:", len(encoded_test))
print("Test windows:", len(test_windows))

Test documents: 35
Test windows: 2161


In [41]:
test_predictions = predict_entities(
    ner_model,
    test_loader,
    encoded_test,
)

# **Saving and creating Zip file**

In [42]:
import zipfile

if set(test_predictions) != set(test_texts):
    raise ValueError("Test document IDs do not match.")

for doc_id, entities in test_predictions.items():
    text_length = len(test_texts[doc_id])

    for start, end, label in entities:
        if not (0 <= start < end <= text_length):
            raise ValueError(f"Invalid span in document {doc_id}")

        if label not in entity_types:
            raise ValueError(f"Invalid label: {label}")

prediction_path = os.path.join(
    PHASE2_OUTPUT_DIR,
    "prediction.json",
)

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

submission_path = os.path.join(
    PHASE2_OUTPUT_DIR,
    "submission.zip",
)

with zipfile.ZipFile(
    submission_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

print("Saved:", prediction_path)
print("Saved:", submission_path)

Saved: /kaggle/working/phase2_hybrid_lora_qkv_ffn_run1/prediction.json
Saved: /kaggle/working/phase2_hybrid_lora_qkv_ffn_run1/submission.zip
